# Age Classification - 5-Fold Cross Validation

**Method:** Video-based split (no overlap between train/test)
**Model:** EfficientNet-B0
**Classes:** Adult / Child

## Step 1: Setup

In [ ]:
!pip install torch torchvision timm scikit-learn seaborn -q

from google.colab import drive
drive.mount('/content/drive')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
from tqdm import tqdm
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using: {device}")

## Step 2: Configuration

In [ ]:
BASE_PATH = Path("/content/drive/MyDrive/face_pipeline_project")
LABELS_FILE = BASE_PATH / "ground_truth/age_labels_v2.csv"
METRICS_FOLDER = BASE_PATH / "metrics"
MODELS_FOLDER = BASE_PATH / "models/cross_validation"

MODELS_FOLDER.mkdir(exist_ok=True, parents=True)

# Training config
N_FOLDS = 5
BATCH_SIZE = 32
EPOCHS = 10
LEARNING_RATE = 1e-4
IMAGE_SIZE = 224

print(f"Labels: {LABELS_FILE}")
print(f"Folds: {N_FOLDS}")
print(f"Epochs per fold: {EPOCHS}")

## Step 3: Load Labels

In [ ]:
df = pd.read_csv(LABELS_FILE)

# Remove duplicates
df = df.drop_duplicates(subset=['image_path'], keep='last')

# Filter Adult and Child only
df = df[df['label'].isin(['adult', 'child'])].reset_index(drop=True)

# Binary labels: 0=child, 1=adult
df['label_int'] = (df['label'] == 'adult').astype(int)

print(f"Total labeled samples: {len(df)}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nVideos: {df['video'].nunique()}")

## Step 4: Create Video-Based Folds

In [ ]:
# Get unique videos
videos = df['video'].unique()
n_videos = len(videos)

print(f"Total videos: {n_videos}")

# Shuffle videos
np.random.seed(42)
np.random.shuffle(videos)

# Split into N_FOLDS groups
fold_size = n_videos // N_FOLDS
video_folds = []

for i in range(N_FOLDS):
    if i == N_FOLDS - 1:
        # Last fold gets remaining videos
        fold_videos = videos[i * fold_size:]
    else:
        fold_videos = videos[i * fold_size:(i + 1) * fold_size]
    video_folds.append(list(fold_videos))
    print(f"Fold {i+1}: {len(fold_videos)} videos")

# Verify no overlap
print(f"\n✓ Video folds created with no overlap")

## Step 5: Dataset Class

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row['image_path']).convert('RGB')
        label = row['label_int']
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("✓ Dataset class ready")

## Step 6: Training Function

In [ ]:
def train_fold(train_df, test_df, fold_num):
    print(f"\n{'='*60}")
    print(f"FOLD {fold_num}")
    print(f"{'='*60}")
    print(f"Train: {len(train_df)} samples | Test: {len(test_df)} samples")
    print(f"Train videos: {train_df['video'].nunique()} | Test videos: {test_df['video'].nunique()}")
    
    # Create datasets
    train_dataset = FaceDataset(train_df, train_transform)
    test_dataset = FaceDataset(test_df, test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    
    # Create model
    model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=2)
    model = model.to(device)
    
    # Class weights
    class_counts = train_df['label_int'].value_counts().sort_index()
    weights = 1.0 / class_counts.values
    weights = weights / weights.sum()
    class_weights = torch.tensor(weights, dtype=torch.float32).to(device)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE)
    
    # Training
    best_acc = 0
    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
        
        # Validation
        model.eval()
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for images, labels in test_loader:
                images = images.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.numpy())
        
        acc = accuracy_score(all_labels, all_preds)
        if acc > best_acc:
            best_acc = acc
            # Save best model
            torch.save(model.state_dict(), MODELS_FOLDER / f"fold_{fold_num}_best.pth")
        
        print(f"  Epoch {epoch+1}/{EPOCHS} - Loss: {train_loss/len(train_loader):.4f} - Acc: {acc:.2%}")
    
    # Load best model for final evaluation
    model.load_state_dict(torch.load(MODELS_FOLDER / f"fold_{fold_num}_best.pth"))
    model.eval()
    
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for images, labels in test_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    
    return all_labels, all_preds, best_acc

print("✓ Training function ready")

## Step 7: Run 5-Fold Cross Validation

In [ ]:
fold_results = []
all_true = []
all_pred = []

for fold_num in range(1, N_FOLDS + 1):
    # Get test videos for this fold
    test_videos = video_folds[fold_num - 1]
    
    # Get train videos (all other folds)
    train_videos = []
    for i, fold_videos in enumerate(video_folds):
        if i != fold_num - 1:
            train_videos.extend(fold_videos)
    
    # Split data
    train_df = df[df['video'].isin(train_videos)].copy()
    test_df = df[df['video'].isin(test_videos)].copy()
    
    # Verify no overlap
    train_video_set = set(train_df['video'].unique())
    test_video_set = set(test_df['video'].unique())
    assert len(train_video_set & test_video_set) == 0, "Overlap detected!"
    
    # Train and evaluate
    y_true, y_pred, acc = train_fold(train_df, test_df, fold_num)
    
    fold_results.append({
        'fold': fold_num,
        'train_samples': len(train_df),
        'test_samples': len(test_df),
        'test_videos': len(test_videos),
        'accuracy': acc
    })
    
    all_true.extend(y_true)
    all_pred.extend(y_pred)
    
    print(f"\n✓ Fold {fold_num} Accuracy: {acc:.2%}")

print("\n" + "="*60)
print("CROSS VALIDATION COMPLETE!")
print("="*60)

## Step 8: Results Summary

In [ ]:
# Create results dataframe
results_df = pd.DataFrame(fold_results)
print(results_df.to_string(index=False))

# Calculate statistics
mean_acc = results_df['accuracy'].mean()
std_acc = results_df['accuracy'].std()
min_acc = results_df['accuracy'].min()
max_acc = results_df['accuracy'].max()

print(f"\n" + "="*40)
print(f"SUMMARY")
print(f"="*40)
print(f"Mean Accuracy: {mean_acc:.2%} ± {std_acc:.2%}")
print(f"Min Accuracy:  {min_acc:.2%}")
print(f"Max Accuracy:  {max_acc:.2%}")

# Save results
results_df.to_csv(METRICS_FOLDER / "cross_validation_results.csv", index=False)
print(f"\n✓ Results saved")

## Step 9: Overall Confusion Matrix

In [ ]:
# Combined confusion matrix from all folds
cm = confusion_matrix(all_true, all_pred, labels=[0, 1])

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Child', 'Adult'],
            yticklabels=['Child', 'Adult'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Age Classification - 5-Fold Cross Validation\nMean Accuracy: {mean_acc:.2%} ± {std_acc:.2%}')
plt.tight_layout()
plt.savefig(METRICS_FOLDER / "confusion_matrix_cross_validation.png", dpi=150)
plt.show()

print("\n" + "="*50)
print(classification_report(all_true, all_pred, target_names=['Child', 'Adult']))

## Step 10: Fold-by-Fold Accuracy Plot

In [ ]:
plt.figure(figsize=(10, 6))
folds = results_df['fold'].values
accs = results_df['accuracy'].values * 100

bars = plt.bar(folds, accs, color='steelblue', edgecolor='black')
plt.axhline(y=mean_acc*100, color='red', linestyle='--', label=f'Mean: {mean_acc:.1%}')

# Add value labels
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
             f'{acc:.1f}%', ha='center', fontsize=12)

plt.xlabel('Fold', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('5-Fold Cross Validation - Age Classification', fontsize=14)
plt.ylim(0, 105)
plt.legend()
plt.tight_layout()
plt.savefig(METRICS_FOLDER / "cross_validation_folds.png", dpi=150)
plt.show()

---
## ✅ Complete!

**Results:**
- 5 models trained (one per fold)
- No video overlap between train/test
- Combined confusion matrix
- Mean accuracy with standard deviation

**Files:**
- Results: `/metrics/cross_validation_results.csv`
- Confusion matrix: `/metrics/confusion_matrix_cross_validation.png`
- Models: `/models/cross_validation/fold_X_best.pth`